# 01 · North Carolina Flood-Event EDA

**StormRoute — Carolina Data Challenge 2026, Natural Science Track**

This notebook is the **Milestone 2 verification gate**. No production modeling,
scoring, API, or website work begins until it has been executed end to end,
reviewed by two teammates, and merged.

It must read as an analytical narrative, not a scratchpad: each section states the
question it tests, answers it with code, and closes with a short conclusion.

| | |
|---|---|
| **Run mode** | `STORMROUTE_DATA_MODE=sample` in CI (tracked fixtures) · `full` in Codespaces |
| **Execute** | `make eda` — restart kernel, run all, fail on any exception |
| **Figures** | exported to `outputs/figures/eda_*.png` with stable names |
| **Report** | findings summarized in `reports/eda/eda_findings.md` |

> **Status:** sections 1–5 implemented. Sections 6–12 are headings only.

## 1. Objective and scope

**Question.** What exactly are we predicting, and what are we deliberately not predicting?

A traveler decides on a route and a departure time before the trip starts. StormRoute
estimates, for every North Carolina county and six-hour window, the probability that a
reported **Flood, Flash Flood, or Debris Flow** event begins there. Route-level exposure is
built from those probabilities later.

The definitions below are read from `configs/`, not typed here, so this notebook cannot
drift away from the pipeline.

In [ ]:
import pandas as pd

from stormroute.config import load_config

DATA = load_config("data")
MODEL = load_config("model")

splits = {name: spec["years"] for name, spec in MODEL["splits"].items() if isinstance(spec, dict)}
pd.DataFrame(
    [
        (
            "Geography",
            f"{DATA['geography']['state']} (state FIPS {DATA['geography']['state_fips']}), "
            f"{DATA['geography']['expected_county_count']} counties",
        ),
        ("Period", f"{DATA['period']['start_year']}–{DATA['period']['end_year']}"),
        (
            "Prediction unit",
            f"county × {DATA['prediction_unit']['window_hours']}-hour window, "
            f"anchored at {DATA['prediction_unit']['window_anchor_utc']} UTC",
        ),
        ("Qualifying hazards", ", ".join(DATA["label"]["hazard_event_types"])),
        ("Positive label", DATA["label"]["onset_rule"]),
        ("Negative label means", DATA["label"]["negative_meaning"]),
        *((f"Split: {name}", ", ".join(map(str, years))) for name, years in splits.items()),
        ("Never a model input", ", ".join(MODEL["features"]["excluded"])),
    ],
    columns=["Definition", "Value"],
).set_index("Definition")

**Conclusion.** The scope is fixed by configuration. Injuries, deaths, and damage are
**descriptive outcomes only**: they are recorded after an event and can never be model
inputs. Sections 3–5 test whether the NOAA records support this scope, and section 9 tests
whether the label can be built.

## 2. Reproducibility information

**Question.** Could a teammate re-run this notebook tomorrow and get the same numbers?

`STORMROUTE_DATA_MODE=sample` runs on the tracked fixture (what CI does). `full` reads the
downloaded NOAA files named in the download record. Figures from a sample run go to
`outputs/figures/sample/`, which is ignored, so they can never overwrite the real ones.

In [ ]:
import hashlib
import json
import platform
import subprocess
import sys
from datetime import UTC, datetime
from importlib.metadata import version

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Markdown, display
from matplotlib.patches import Patch

from stormroute.config import REPO_ROOT, data_mode, data_path
from stormroute.data.noaa import (
    RAW_COLUMNS,
    RECORD_FILENAME,
    active_detail_files,
    clean_events,
    read_details,
)

MODE = data_mode()
SEED = MODEL["final_model"]["random_seed"]
FIGURES = (
    REPO_ROOT / "outputs" / "figures"
    if MODE == "full"
    else REPO_ROOT / "outputs" / "figures" / "sample"
)

if MODE == "sample":
    SOURCE_PATHS = [data_path("sample") / "storm_events_sample.csv"]
    sources = pd.DataFrame(
        {
            "file": [p.name for p in SOURCE_PATHS],
            "sha256": [hashlib.sha256(p.read_bytes()).hexdigest()[:16] for p in SOURCE_PATHS],
        }
    )
else:
    period = range(DATA["period"]["start_year"], DATA["period"]["end_year"] + 1)
    SOURCE_PATHS = active_detail_files(data_path("noaa_raw"), period)
    record = json.loads((data_path("noaa_raw") / RECORD_FILENAME).read_text())
    sources = pd.DataFrame(
        [
            {
                "year": record["files"][p.name]["year"],
                "file": p.name,
                "sha256": record["files"][p.name]["sha256"][:16],
                "recorded_at_utc": record["files"][p.name]["recorded_at_utc"],
            }
            for p in SOURCE_PATHS
        ]
    )

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    commit = "unavailable"

manifest = yaml.safe_load((REPO_ROOT / "data" / "data_manifest.yaml").read_text())
print(f"Analysis run (UTC): {datetime.now(UTC):%Y-%m-%d %H:%M}")
print(f"Data mode:          {MODE}")
print(f"Git commit:         {commit}")
print(
    f"Data manifest:      version {manifest['manifest_version']}, "
    f"updated {manifest['last_updated']}"
)
print(f"Python:             {sys.version.split()[0]} on {platform.system()}")
print(
    "Packages:           "
    + ", ".join(
        f"{pkg} {version(pkg)}"
        for pkg in ("stormroute", "pandas", "numpy", "matplotlib", "pyarrow")
    )
)
print(f"Random seed:        {SEED}")
print(f"Figures written to: {FIGURES.relative_to(REPO_ROOT)}")
sources

In [ ]:
# Figure standard: validated palette, recessive chrome, source note on every figure.
COLORS = {
    "surface": "#fcfcfb",
    "ink": "#0b0b0b",
    "ink_secondary": "#52514e",
    "muted": "#898781",
    "grid": "#e1e0d9",
    "baseline": "#c3c2b7",
    "accent": "#2a78d6",  # categorical slot 1: the thing the figure is about
    "context": "#898781",  # neutral for context marks
}
mpl.rcParams.update(
    {
        "figure.facecolor": COLORS["surface"],
        "axes.facecolor": COLORS["surface"],
        "savefig.facecolor": COLORS["surface"],
        "font.family": "sans-serif",
        "font.size": 11,
        "text.color": COLORS["ink"],
        "axes.labelcolor": COLORS["ink_secondary"],
        "xtick.color": COLORS["muted"],
        "ytick.color": COLORS["ink_secondary"],
        "axes.edgecolor": COLORS["baseline"],
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "grid.color": COLORS["grid"],
        "grid.linewidth": 0.8,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
    }
)
SOURCE_NOTE = "Source: NOAA NCEI Storm Events Database, details files " + (
    "(tracked sample fixture)"
    if MODE == "sample"
    else f"{DATA['period']['start_year']}–{DATA['period']['end_year']}"
)


def save_figure(fig: plt.Figure, name: str) -> None:
    FIGURES.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES / f"{name}.png", dpi=200, bbox_inches="tight")
    print(f"saved {(FIGURES / f'{name}.png').relative_to(REPO_ROOT)}")


def conclude(text: str) -> None:
    display(Markdown(f"**Conclusion.** {text}"))


def n(count: int, noun: str, plural: str | None = None) -> str:
    return f"{count:,} {noun if count == 1 else plural or noun + 's'}"


def pct(part: float, whole: float) -> str:
    return f"{100 * part / whole:.1f}%" if whole else "n/a"

**Conclusion.** Every number below traces to the files and checksums listed above, the
commit printed with them, and a fixed seed. Paths resolve from the repository root.

## 3. Schema and first-pass inspection

**Question.** Is the raw data shaped the way the NCEI documentation says it is?

Everything is read as **text**, exactly as published, so no county code or time loses a
leading zero to number parsing. The analysis reads a subset of the published columns.
Narratives are skipped because they are large and are never a model input.

In [ ]:
published_columns = pd.read_csv(SOURCE_PATHS[0], nrows=0).columns.tolist()
ANALYSIS_COLUMNS = [*RAW_COLUMNS, "BEGIN_DATE_TIME", "END_DATE_TIME", "BEGIN_LAT", "BEGIN_LON"]
raw = read_details(SOURCE_PATHS, ANALYSIS_COLUMNS)
raw_nc = raw[raw["STATE_FIPS"].str.zfill(2) == DATA["geography"]["state_fips"]]

print(f"Published columns: {len(published_columns)}  |  analysed: {len(ANALYSIS_COLUMNS)}")
print(f"Rows, all states:  {len(raw):,}  |  North Carolina: {len(raw_nc):,}")
print(f"Column dtypes:     {raw.dtypes.astype(str).value_counts().to_dict()} (all text by design)")

In [ ]:
raw_nc[
    [
        "EVENT_ID",
        "EVENT_TYPE",
        "CZ_TYPE",
        "CZ_FIPS",
        "CZ_NAME",
        "BEGIN_DATE_TIME",
        "END_DATE_TIME",
        "CZ_TIMEZONE",
        "DAMAGE_PROPERTY",
    ]
].sample(min(5, len(raw_nc)), random_state=SEED)

In [ ]:
duplicate_ids = int(raw["EVENT_ID"].duplicated().sum())
exact_duplicates = int(raw.duplicated().sum())
pd.DataFrame(
    [
        ("Duplicated EVENT_ID values", duplicate_ids),
        ("...of which exact duplicate rows", exact_duplicates),
        ("Distinct STATE values", raw["STATE"].nunique()),
        ("Distinct EVENT_TYPE values (all states)", raw["EVENT_TYPE"].nunique()),
        ("Distinct EVENT_TYPE values (NC)", raw_nc["EVENT_TYPE"].nunique()),
    ],
    columns=["Check", "Value"],
).set_index("Check")

In [ ]:
pd.crosstab(raw_nc["CZ_TYPE"], "NC rows").rename(
    index={"C": "C: county", "Z": "Z: forecast zone", "M": "M: marine"}
)

### Missingness in the North Carolina source rows

**Question.** Which source fields are blank before hazard filtering and cleaning?

Count null, empty, and whitespace-only values across all NC source rows, including
repeated records and non-target hazards. Each field uses the same row denominator.
A literal zero is observed data, not missing. Invalid nonblank values are a separate
validation problem. Coordinate availability can depend on event type; this table
alone cannot establish whether a field is suitable for modeling.


In [ ]:
missing_mask = raw_nc.apply(lambda column: column.isna() | column.str.strip().eq(""))
missingness = pd.DataFrame(
    {
        "field": raw_nc.columns,
        "rows": len(raw_nc),
        "missing_count": missing_mask.sum().to_numpy(dtype=int),
    }
)
missingness["missing_fraction"] = (
    missingness["missing_count"] / len(raw_nc) if len(raw_nc) else np.nan
)
missingness = missingness.sort_values(
    ["missing_count", "field"], ascending=[False, True], ignore_index=True
)
assert missingness["missing_count"].between(0, len(raw_nc)).all()
display(missingness)

metrics_dir = REPO_ROOT / "outputs" / "metrics"
if MODE == "sample":
    metrics_dir = metrics_dir / "sample"
metrics_dir.mkdir(parents=True, exist_ok=True)
missingness_payload = {
    "data_mode": MODE,
    "population": "North Carolina source rows before hazard filtering and deduplication",
    "missing_definition": "null, empty, or whitespace-only; literal zero is not missing",
    "source_files": [p.name for p in SOURCE_PATHS],
    "source_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in SOURCE_PATHS},
    "git_commit": commit,
    "rows": len(raw_nc),
    "fields": json.loads(missingness.to_json(orient="records")),
}
metrics_path = metrics_dir / "eda_missingness.json"
metrics_path.write_text(
    json.dumps(missingness_payload, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print(f"Saved {metrics_path.relative_to(REPO_ROOT)}")

fig, ax = plt.subplots(figsize=(10, max(5, len(missingness) * 0.25)))
ax.barh(
    missingness["field"], missingness["missing_fraction"].fillna(0) * 100, color=COLORS["accent"]
)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Missing source rows (%)")
ax.set_ylabel("Source field")
ax.set_title(f"Missing fields in NC source rows ({MODE} mode; n={len(raw_nc):,})")
ax.xaxis.grid(True)
ax.set_axisbelow(True)
if not len(raw_nc):
    ax.text(0.5, 0.5, "No NC rows: percentages undefined", transform=ax.transAxes, ha="center")
fig.text(0.01, -0.02, SOURCE_NOTE + ". Before hazard filtering and deduplication.", fontsize=9)
fig.tight_layout()
save_figure(fig, "eda_missingness")
display(fig)
plt.close(fig)

fields_with_missing = int((missingness["missing_count"] > 0).sum())
conclude(
    f"{fields_with_missing} of {len(missingness)} inspected fields contain blanks across "
    f"{len(raw_nc):,} NC source rows. "
    + ("No NC rows were available; missing fractions are undefined. " if not len(raw_nc) else "")
    + (
        "These synthetic fixture results are a smoke test, not NC population estimates. "
        if MODE == "sample"
        else ""
    )
    + "Missingness does not measure invalid nonblank values or prove predictor suitability."
)

In [ ]:
if duplicate_ids == 0:
    identity = "every EVENT_ID is unique"
elif duplicate_ids == exact_duplicates:
    identity = (
        f"{n(duplicate_ids, 'EVENT_ID')} repeat, and every repeat is an exact duplicate row, "
        "which ingestion removes"
    )
else:
    identity = (
        f"{n(duplicate_ids - exact_duplicates, 'EVENT_ID')} repeat with **different** content, "
        "which ingestion rejects"
    )
conclude(
    f"Across {n(len(SOURCE_PATHS), 'source file')} there are {len(raw):,} rows, "
    f"{len(raw_nc):,} of them in North Carolina, and {identity}. NCEI publishes county "
    "codes, days, and times **unpadded** (e.g. `21` for Buncombe), so ingestion pads every "
    "one explicitly."
)

## 4. Time-field validation

**Question.** Can every event be placed on one unambiguous UTC timeline?

**Conversion rule.** NCEI splits each timestamp into `YEARMONTH`, `DAY`, and `TIME`, all
unpadded, and records a fixed offset in `CZ_TIMEZONE` (e.g. `EST-5`). Ingestion applies that
offset as written, with no daylight-saving lookup: `UTC = local − offset`.

The timestamps are then checked against NCEI's own `BEGIN_DATE_TIME` and `END_DATE_TIME`
text fields, an independent rendering of the same instant.

In [ ]:
events, report = clean_events(raw, source_files=[p.name for p in SOURCE_PATHS])
pd.DataFrame(
    [
        ("Events after all filters", report.events_out),
        ("Excluded: begin time invalid", report.excluded_invalid_begin),
        ("Excluded: end time invalid", report.excluded_invalid_end),
        ("Excluded: end before begin", report.excluded_end_before_begin),
        ("Kept: end time missing (labels need only begin)", report.missing_end_kept),
        ("Kept: zero duration (end = begin)", report.zero_duration),
    ],
    columns=["Timestamp check", "Events"],
).set_index("Timestamp check")

In [ ]:
published = (
    raw_nc.drop_duplicates("EVENT_ID")
    .set_index("EVENT_ID")
    .loc[events["event_id"], ["BEGIN_DATE_TIME", "END_DATE_TIME", "CZ_TIMEZONE"]]
)
offset = pd.to_timedelta(published["CZ_TIMEZONE"].str.extract(r"-(\d+)$")[0].astype(int), unit="h")


def as_local(stamps: pd.Series) -> pd.Series:
    return pd.to_datetime(stamps.replace("", None), format="%d-%b-%y %H:%M:%S")


begin_local = as_local(published["BEGIN_DATE_TIME"]).to_numpy()
end_local = as_local(published["END_DATE_TIME"]).to_numpy()
ours_begin = (events["begin_utc"].dt.tz_localize(None) - offset.to_numpy()).to_numpy()
ours_end = (events["end_utc"].dt.tz_localize(None) - offset.to_numpy()).to_numpy()

begin_mismatch = int((begin_local != ours_begin).sum())
end_both = ~pd.isna(end_local) & ~pd.isna(ours_end)
end_mismatch = int((end_local[end_both] != ours_end[end_both]).sum())
print(f"Begin times disagreeing with BEGIN_DATE_TIME: {begin_mismatch} of {len(events):,}")
print(f"End times disagreeing with END_DATE_TIME:     {end_mismatch} of {int(end_both.sum()):,}")
print(f"CZ_TIMEZONE values on NC rows: {raw_nc['CZ_TIMEZONE'].value_counts().to_dict()}")

In [ ]:
local_begin = pd.Series(begin_local, index=events.index)
local_end = pd.Series(end_local, index=events.index)
on_boundary = (events["begin_utc"].dt.minute == 0) & (events["begin_utc"].dt.hour % 6 == 0)
CASES = {
    "Crosses local midnight": (local_end.dt.date != local_begin.dt.date) & local_end.notna(),
    "Crosses UTC midnight": (events["end_utc"].dt.date != events["begin_utc"].dt.date)
    & events["end_utc"].notna(),
    "Crosses a UTC year boundary": events["end_utc"].dt.year > events["begin_utc"].dt.year,
    "Begins exactly on a six-hour UTC boundary": on_boundary,
    "Zero duration on a six-hour UTC boundary": on_boundary
    & (events["end_utc"] == events["begin_utc"]),
}
pd.DataFrame({"Events": {case: int(mask.sum()) for case, mask in CASES.items()}}).rename_axis(
    "Case"
)

### Open question: standard time or clock time?

Every North Carolina record says `EST-5`, including summer events. If NCEI's summer times
are actually clock time (EDT, UTC−4), then those events are **one hour late** in UTC. The
field text alone cannot settle this, so the size of the exposure is measured here and the
answer comes from comparing events against their NWS warning timestamps.

In [ ]:
in_dst = (
    events["begin_utc"].dt.tz_convert("America/New_York").map(lambda t: t.dst() != pd.Timedelta(0))
)
print(
    f"Events beginning while Eastern daylight time is in force: {int(in_dst.sum()):,} "
    f"of {len(events):,} ({pct(in_dst.sum(), len(events))})"
)

### Spot-check candidates

The gate requires at least ten events checked **by a person** against NOAA's event pages and
NWS products. These are chosen deterministically to cover the riskiest cases, with three
daylight-saving-period events to settle the question above. In full mode they are written to
`reports/eda/spot_check_candidates.csv`, with blank columns for the reviewer.

In [ ]:
durations = events["end_utc"] - events["begin_utc"]
helene = events["begin_utc"].between(
    pd.Timestamp("2024-09-25", tz="UTC"), pd.Timestamp("2024-09-29", tz="UTC")
)
strata = [
    ("Daylight-saving period (settles EST-5)", in_dst, 3),
    ("Standard-time period", ~in_dst, 1),
    ("Zero duration", durations == pd.Timedelta(0), 1),
    ("Crosses local midnight", CASES["Crosses local midnight"], 1),
    ("Debris Flow", events["event_type"] == "Debris Flow", 1),
    ("Helene period", helene, 1),
]
chosen: list[tuple[str, str]] = []
for label, mask, size in strata:
    pool = events[mask & ~events["event_id"].isin([e for e, _ in chosen])]
    chosen += [(e, label) for e in pool.sample(min(size, len(pool)), random_state=SEED)["event_id"]]
remaining = events[~events["event_id"].isin([e for e, _ in chosen])]
if len(remaining):
    chosen.append(
        (remaining.loc[durations[remaining.index].idxmax(), "event_id"], "Longest duration")
    )
remaining = events[~events["event_id"].isin([e for e, _ in chosen])]
fill = max(0, 10 - len(chosen))
chosen += [
    (e, "Random")
    for e in remaining.sample(min(fill, len(remaining)), random_state=SEED)["event_id"]
]

picked = events.set_index("event_id").loc[[e for e, _ in chosen]]
spot_check = pd.DataFrame(
    {
        "event_id": picked.index,
        "why_selected": [label for _, label in chosen],
        "event_type": picked["event_type"].astype(str).to_numpy(),
        "county_fips": picked["county_fips"].to_numpy(),
        "begin_as_published": published.loc[picked.index, "BEGIN_DATE_TIME"].to_numpy(),
        "begin_utc": picked["begin_utc"].dt.strftime("%Y-%m-%d %H:%M").to_numpy(),
        "ncei_event_page": [
            f"https://www.ncdc.noaa.gov/stormevents/eventdetails.jsp?id={e}" for e in picked.index
        ],
        "reviewer": "",
        "county_matches": "",
        "utc_time_matches_nws_product": "",
        "notes": "",
    }
)
if MODE == "full":
    target = REPO_ROOT / "reports" / "eda" / "spot_check_candidates.csv"
    spot_check.to_csv(target, index=False)
    print(f"written {target.relative_to(REPO_ROOT)}")
spot_check.iloc[:, :6]

In [ ]:
bad_end = report.excluded_invalid_end + report.excluded_end_before_begin
checked_ends = int(end_both.sum())
zero_on_boundary = int(CASES["Zero duration on a six-hour UTC boundary"].sum())
dst_events = int(in_dst.sum())
conclude(
    f"All {len(events):,} qualifying events have a valid begin time; "
    f"{n(bad_end, 'event')} had to be excluded for a bad end time. "
    "Ingestion's UTC timestamps agree with NCEI's own date-time text for "
    f"{len(events) - begin_mismatch:,} of {len(events):,} begin times and "
    f"{checked_ends - end_mismatch:,} of {checked_ends:,} end times. "
    f"{n(report.zero_duration, 'event')} ({pct(report.zero_duration, len(events))}) "
    f"have zero duration, and {zero_on_boundary:,} of them begin exactly on a window boundary: "
    "the onset rule places each in exactly one window. **Unresolved:** whether `EST-5` is "
    f"true for the {n(dst_events, 'daylight-saving-period event')} "
    f"({pct(dst_events, len(events))}). The spot check settles it."
)

## 5. Hazard and geography filters

**Question.** How many records survive each filter, what does the hazard filter leave out,
and can every event be placed in a county?

The filters run in `stormroute.data.noaa.clean_events`, which counts every excluded row.

In [ ]:
qualifying_rows = report.rows_qualifying_type
placeable = (
    qualifying_rows
    - report.exact_duplicates_removed
    - sum(report.excluded_unresolved_geography.values())
)
funnel = pd.DataFrame(
    [
        ("All states, all event types", len(raw)),
        (f"{DATA['geography']['state']} only", report.rows_in_state),
        ("Qualifying hazard types only", qualifying_rows),
        ("Exact duplicates removed", qualifying_rows - report.exact_duplicates_removed),
        ("County-coded (placeable in a county)", placeable),
        ("Valid timestamps: final events", report.events_out),
    ],
    columns=["Stage", "Records"],
).set_index("Stage")
funnel["Share of previous"] = [""] + [
    pct(b, a) for a, b in zip(funnel["Records"][:-1], funnel["Records"][1:], strict=True)
]
funnel

In [ ]:
qualifying_nc = raw_nc[
    raw_nc["EVENT_TYPE"].isin(DATA["label"]["hazard_event_types"])
].drop_duplicates()
geo = pd.DataFrame(
    [
        ("County-coded (CZ_TYPE = C)", int((qualifying_nc["CZ_TYPE"] == "C").sum())),
        ("Zone-coded (CZ_TYPE = Z)", int((qualifying_nc["CZ_TYPE"] == "Z").sum())),
        ("Has begin coordinates", int((qualifying_nc["BEGIN_LAT"] != "").sum())),
        ("Cannot be assigned to a county", sum(report.excluded_unresolved_geography.values())),
    ],
    columns=["Qualifying NC records", "Count"],
).set_index("Qualifying NC records")
geo

### What the hazard filter leaves out

The locked definition keeps exactly three event types. Several other water-related types
also appear in North Carolina. Grouping them as "water-related" is an analyst choice for
this display only; it does not change the filter.

In [ ]:
WATER_RELATED = [
    "Flash Flood",
    "Flood",
    "Debris Flow",
    "Heavy Rain",
    "Tropical Storm",
    "Coastal Flood",
    "Hurricane (Typhoon)",
    "Storm Surge/Tide",
    "Tropical Depression",
]
KEPT = DATA["label"]["hazard_event_types"]
DISCUSSED = {"Heavy Rain", "Coastal Flood"}  # excluded types the gate decision must address

nc_types = raw_nc.drop_duplicates()["EVENT_TYPE"].value_counts()
plot = pd.DataFrame(
    {"type": WATER_RELATED, "reports": [int(nc_types.get(t, 0)) for t in WATER_RELATED]}
)
plot = plot[plot["reports"] > 0].assign(kept=lambda d: d["type"].isin(KEPT))
plot = pd.concat(
    [
        plot[plot["kept"]].sort_values("reports", ascending=False),
        plot[~plot["kept"]].sort_values("reports", ascending=False),
    ]
)

fig, ax = plt.subplots(figsize=(10, 5.6))
y = np.arange(len(plot))[::-1]
ax.barh(
    y,
    plot["reports"],
    height=0.62,
    edgecolor=COLORS["surface"],
    linewidth=2,
    color=np.where(plot["kept"], COLORS["accent"], COLORS["context"]),
)
ax.set_yticks(y, plot["type"])
ax.tick_params(axis="y", length=0)
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)
ax.set_xlim(0, plot["reports"].max() * 1.12)
ax.xaxis.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))
too_short_to_read = plot["reports"].max() * 0.03
for yi, row in zip(y, plot.itertuples(), strict=True):
    if row.kept or row.type in DISCUSSED or row.reports < too_short_to_read:
        ax.text(
            row.reports + plot["reports"].max() * 0.012,
            yi,
            f"{row.reports:,}",
            va="center",
            color=COLORS["ink_secondary"],
            fontsize=10,
        )
ax.set_xlabel(f"Reports in North Carolina, {SOURCE_NOTE.split('files ')[-1]} (count)")
ax.set_title(
    f"The hazard filter keeps {int(plot['kept'].sum())} of {len(plot)} water-related event types",
    pad=28,
)
ax.text(
    0,
    1.02,
    "NOAA Storm Events reports by type; the label uses only the kept types",
    transform=ax.transAxes,
    color=COLORS["ink_secondary"],
    fontsize=11,
)
ax.legend(
    handles=[
        Patch(color=COLORS["accent"], label="Kept: qualifying hazard"),
        Patch(color=COLORS["context"], label="Excluded"),
    ],
    loc="lower right",
    frameon=False,
)
fig.text(
    0.01,
    -0.02,
    SOURCE_NOTE + ". Water-related grouping is for display only.",
    color=COLORS["muted"],
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "eda_event_type_counts")
display(fig)
plt.close(fig)

In [ ]:
excluded = plot[~plot["kept"]].set_index("type")["reports"]
unresolved = sum(report.excluded_unresolved_geography.values())
geography = (
    "is county-coded, so **no** event had to be dropped for unplaceable geography, and a "
    "zone-to-county crosswalk is unnecessary for this label."
    if unresolved == 0
    else f"except {n(unresolved, 'zone-coded record')} is county-coded; "
    "those are excluded and counted."
)
conclude(
    f"{report.events_out:,} of {len(raw):,} records survive ({pct(report.events_out, len(raw))}). "
    f"Every qualifying record {geography} "
    f"The filter leaves out {int(excluded.sum()):,} water-related reports, notably Heavy Rain "
    f"({int(excluded.get('Heavy Rain', 0)):,}) and Coastal Flood "
    f"({int(excluded.get('Coastal Flood', 0)):,}). Heavy Rain is excluded by ADR 0000. "
    "**The team still needs to confirm the Coastal Flood exclusion** at the gate."
)

## 6. Temporal coverage

**Question.** How do retained event counts vary across the configured years and months?

Annual and pooled-month counts and figures are implemented below, along with
county and hazard breakdowns for Cameron's count task. Full-data interpretation,
duration distributions, reporting-gap analysis, and a Helene-period comparison
remain pending. This section does not yet establish reporting stability.


### Event counts (Cameron)

**Question.** How many retained qualifying reports begin in each year, month,
county, and hazard category?

Use the cleaned, deduplicated `events` table from section 4. Count each event ID
once per breakdown, using its UTC beginning time. Restrict beginnings to the
configured 2015?2024 UTC period and report exclusions explicitly: a source-year
file can contain a local December event whose UTC beginning falls in January.
Months are pooled across years, not monthly rates. Zero-filled year and month
bins mean no retained reports in these inputs, not proof of complete coverage.
County totals include observed FIPS only; identifying all zero-report counties
requires the county-boundary inventory and remains a separate task.


In [ ]:
start_year = DATA["period"]["start_year"]
end_year = DATA["period"]["end_year"]
period_start = pd.Timestamp(year=start_year, month=1, day=1, tz="UTC")
period_end = pd.Timestamp(year=end_year + 1, month=1, day=1, tz="UTC")
in_period = events["begin_utc"].ge(period_start) & events["begin_utc"].lt(period_end)
counted_events = events.loc[in_period].copy()
assert counted_events["event_id"].is_unique

annual_counts = (
    counted_events["begin_utc"]
    .dt.year.value_counts()
    .reindex(range(start_year, end_year + 1), fill_value=0)
    .sort_index()
    .rename_axis("year")
    .rename("event_count")
    .reset_index()
)
monthly_counts = (
    counted_events["begin_utc"]
    .dt.month.value_counts()
    .reindex(range(1, 13), fill_value=0)
    .sort_index()
    .rename_axis("month")
    .rename("event_count")
    .reset_index()
)
county_counts = (
    counted_events.groupby("county_fips", observed=True)
    .size()
    .rename("event_count")
    .reset_index()
    .sort_values(["event_count", "county_fips"], ascending=[False, True], ignore_index=True)
)
hazard_counts = (
    counted_events["event_type"]
    .value_counts()
    .reindex(DATA["label"]["hazard_event_types"], fill_value=0)
    .rename_axis("event_type")
    .rename("event_count")
    .reset_index()
)
count_tables = {
    "year": annual_counts,
    "month": monthly_counts,
    "county": county_counts,
    "hazard": hazard_counts,
}
for dimension, table in count_tables.items():
    assert int(table["event_count"].sum()) == len(counted_events)
    display(Markdown(f"**Counts by {dimension}**"))
    display(table)

count_payload = {
    "data_mode": MODE,
    "population": "Unique cleaned NC qualifying events with UTC beginnings in configured period",
    "time_basis": "UTC beginning; months pooled across years",
    "period_start_inclusive": period_start.isoformat(),
    "period_end_exclusive": period_end.isoformat(),
    "cleaned_events": len(events),
    "events_counted": len(counted_events),
    "excluded_outside_utc_period": int((~in_period).sum()),
    "county_coverage": "Observed county FIPS only; no complete county inventory inferred",
    "source_files": [p.name for p in SOURCE_PATHS],
    "source_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in SOURCE_PATHS},
    "git_commit": commit,
    "counts": {
        name: json.loads(table.to_json(orient="records")) for name, table in count_tables.items()
    },
}
count_path = metrics_dir / "eda_event_counts.json"
count_path.write_text(json.dumps(count_payload, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(f"Saved {count_path.relative_to(REPO_ROOT)}")

In [ ]:
for table, column, title, xlabel, filename in [
    (
        annual_counts,
        "year",
        "Qualifying event reports by UTC start year",
        "UTC start year",
        "eda_events_by_year",
    ),
    (
        monthly_counts,
        "month",
        "Qualifying event reports by UTC start month",
        "UTC start month (pooled across years)",
        "eda_monthly_seasonality",
    ),
]:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(table[column], table["event_count"], color=COLORS["accent"])
    ax.set_xticks(table[column])
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Retained unique event reports (count)")
    ax.set_title(f"{title} ({MODE} mode)")
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
    fig.text(
        0.01,
        -0.04,
        SOURCE_NOTE + ". Counts are not hazard probabilities or coverage estimates.",
        fontsize=9,
    )
    fig.tight_layout()
    save_figure(fig, filename)
    display(fig)
    plt.close(fig)

conclude(
    f"All four breakdowns reconcile to {len(counted_events):,} retained event reports; "
    f"{int((~in_period).sum()):,} cleaned events fall outside the UTC analysis period. "
    f"{len(county_counts)} county FIPS appear in these inputs. "
    + (
        "Synthetic sample counts cannot establish seasonality, reporting trends, or county risk. "
        if MODE == "sample"
        else ""
    )
    + "These are report counts, not positive county-window rates. Reporting completeness, "
    "county-size effects, duration, and Helene-specific analysis remain unverified."
)

## 7. Geographic distribution

**Question.** Where do reported events occur, and how much of that is county size or reporting bias?

**This section must:**

- events by county
- normalized event rate by county and year
- North Carolina choropleth (export `eda_county_choropleth.png`)
- counties with zero reported qualifying events
- discussion of reporting bias and county-size effects

**Conclusion.** _TBD — state what the result shows, in one or two sentences, after the code runs._

<!-- TODO(milestone-2): add analysis cells below this heading. -->

## 8. Descriptive impact analysis

**Question.** Why does this matter to a traveler — stated without sensationalizing harm?

**This section must:**

- injury, death, and property-damage distributions
- median and upper-tail damage by event type
- an explicit warning that these variables occur after the event and **cannot be predictors**
- one chart connecting the project to traveler safety

**Conclusion.** _TBD — state what the result shows, in one or two sentences, after the code runs._

<!-- TODO(milestone-2): add analysis cells below this heading. -->

## 9. Label construction experiment

**Question.** Can the locked label actually be built, and how rare is the positive class?

**This section must:**

- generate all county × six-hour windows for at least one complete year
- assign labels with the locked onset rule (`window_start <= begin < window_end`)
- recompute with the specification's overlap rule and report the difference, including zero-duration events and ongoing multi-window events
- calculate the positive rate
- demonstrate why event rows alone cannot train a classifier
- class imbalance by month and county (export `eda_class_imbalance.png`)

**Conclusion.** _TBD — state what the result shows, in one or two sentences, after the code runs._

<!-- TODO(milestone-2): add analysis cells below this heading. -->

## 10. Weather-feature feasibility check

**Question.** Are the weather features available, consistent, and alignable to event times?

**This section must:**

- retrieve a small ERA5-Land / Open-Meteo sample for a mountain, a Piedmont, and a coastal county
- verify hourly precipitation, soil moisture, temperature, and wind availability
- test 3-, 6-, 24-, and 72-hour rolling precipitation
- compare feature timestamps with event timestamps (export `eda_precipitation_event_comparison.png`)

**Conclusion.** _TBD — state what the result shows, in one or two sentences, after the code runs._

<!-- TODO(milestone-2): add analysis cells below this heading. -->

## 11. Leakage audit

**Question.** Could any candidate feature smuggle in information from after departure?

**This section must:**

- list every candidate feature
- classify each: known before departure / contemporaneous / known only after the event
- reject injuries, deaths, damage, narratives, event end time, and future-derived aggregates
- state the reanalysis-versus-live-forecast limitation

**Conclusion.** _TBD — state what the result shows, in one or two sentences, after the code runs._

<!-- TODO(milestone-2): add analysis cells below this heading. -->

## 12. EDA conclusions and model decisions

The gate decision. Each answer must be backed by a number or a figure above.

| # | Question | Answer | Evidence |
|---|---|---|---|
| 1 | Is the proposed label constructible? | _TBD_ | |
| 2 | Is the positive class large enough for the proposed temporal splits? | _TBD_ | |
| 3 | Which features are available consistently? | _TBD_ | |
| 4 | Which fields are excluded, and why? | _TBD_ | |
| 5 | Which data-quality risks remain? | _TBD_ | |
| 6 | **Does the team proceed, change scope, or stop?** | _TBD_ | |

**Decision recorded by:** _TBD (Mason)_ · **Reviewed by:** _TBD (Econ/Stats)_, _TBD (CS)_

<!-- TODO(milestone-2): complete after the full-data run. -->